# Task 2 — Teaching a computer to read sentiment

In the first activity you sorted film reviews into **positive** and **negative** by hand, using lists of "good" and "bad" words.

Now we'll get a computer to do the same job — in **two different ways**:

1. **The rule-based model** — the method you used earlier (counting good and bad words).
2. **The "meaning" model** — it turns each sentence into numbers and compares them.

Then we'll see where each one succeeds and where it gets fooled.

> Everything in this notebook uses only Python's built-in tools — **nothing to install**.

---

### How to use this notebook
- Run each grey **code cell** by clicking it and pressing **▶** (or `Shift + Enter`).
- Run the cells **in order, top to bottom.**
- You don't need to understand every line. The parts you'll actually change are clearly marked with
  `# 👉 CHANGE THIS`.
- If you see a red error, try to re-run the cell above it, then try again.

## Step 1 — Model 1: the rule-based classifier

This is what we did on paper earlier. We give the computer two lists —
positive words and negative words — and it decides by counting which kind appears more.

Run the cell below to build it.

In [ ]:
# Our two word lists
positive_words = [
    "good", "great", "brilliant", "amazing", "wonderful", "love", "loved",
    "funny", "clever", "beautiful", "excellent", "perfect", "enjoyed",
    "gripping", "superb", "heartwarming", "exciting", "best", "recommend"
]

negative_words = [
    "bad", "boring", "terrible", "awful", "dull", "predictable", "weak",
    "poor", "dreadful", "disappointing", "forgettable", "slow", "mess",
    "waste", "lifeless", "lazy", "worst", "hate", "hated"
]

def rule_based_sentiment(review):
    """Counts positive vs negative words and picks the bigger side."""
    text = review.lower()                 # ignore capital letters
    words = text.split()                  # break the sentence into words
    pos_count = sum(word.strip(".,!?") in positive_words for word in words)
    neg_count = sum(word.strip(".,!?") in negative_words for word in words)

    if pos_count > neg_count:
        return "POSITIVE"
    elif neg_count > pos_count:
        return "NEGATIVE"
    else:
        return "UNSURE"

print("✅ Rule-based model ready.")

: 

### Try it out

Run this cell to test the rule-based model on a review.
Then **change the sentence** and run it again to see what happens. Try a few different sentences and try to break the rule-based method.

In [ ]:
review = "A brilliant film with wonderful acting"   # 👉 CHANGE THIS

result = rule_based_sentiment(review)
print(f'Review:  "{review}"')
print(f'Verdict: {result}')

## Step 2 — Model 2: comparing by *meaning*

The rule-based model has a big weakness: it only knows the exact words on its lists.
If a review says *"magnificent"* and that word isn't on the list, the model is stuck.

Our second model works differently. Instead of a fixed list, we give it a few **example**
positive reviews and a few **example** negative ones. To classify a new review, it asks:

> *Does this review share more words with the positive examples, or the negative ones?*

To do that, we turn every sentence into a **vector** — a list of numbers counting how often each
word appears. Sentences that use similar words end up with similar vectors. This is the same idea
as the *"words as numbers"* trick that real AI models use, just in a simple form we can build ourselves.

Run the cell below to build the tools.

In [ ]:
import math
from collections import Counter

def tokenise(text):
    """Turn a sentence into a clean list of lowercase words."""
    text = text.lower()
    # replace anything that isn't a letter with a space, then split
    cleaned = "".join(ch if ch.isalpha() or ch == " " else " " for ch in text)
    return cleaned.split()

def make_vector(text):
    """Turn a sentence into a word-count 'fingerprint'."""
    return Counter(tokenise(text))

def cosine_similarity(vec_a, vec_b):
    """Measure how similar two word-fingerprints are (0 = nothing in common, 1 = identical)."""
    shared_words = set(vec_a) & set(vec_b)
    dot = sum(vec_a[w] * vec_b[w] for w in shared_words)
    size_a = math.sqrt(sum(count * count for count in vec_a.values()))
    size_b = math.sqrt(sum(count * count for count in vec_b.values()))
    if size_a == 0 or size_b == 0:
        return 0.0
    return dot / (size_a * size_b)

print("✅ Tools ready.")

### See a fingerprint

Before we classify anything, let's peek at what `make_vector` does. Run this to see a
sentence turned into word-counts.

In [ ]:
example = "A good, good film — a great film"   # 👉 CHANGE THIS
print(make_vector(example))

### Set up the examples and the classifier

Now we give the model its example reviews and write the function that compares a new review
against them.

In [ ]:
# A few clear examples of each sentiment. The model compares new reviews against these.
positive_examples = [
    "I loved this film, it was fantastic",
    "A wonderful, moving and beautiful movie",
    "Brilliant and enjoyable from start to finish",
    "A great and exciting story with superb acting",
]
negative_examples = [
    "I hated this film, it was rubbish",
    "A dull, disappointing and boring movie",
    "Terrible and a complete waste of time",
    "A weak and lazy film with poor acting",
]

# Turn every example into its word-count fingerprint.
positive_vectors = [make_vector(s) for s in positive_examples]
negative_vectors = [make_vector(s) for s in negative_examples]

def similarity_sentiment(review):
    """Compare the review to the positive and negative examples by shared words."""
    review_vec = make_vector(review)

    # Average similarity to the positive examples, and to the negative ones.
    pos_score = sum(cosine_similarity(review_vec, v) for v in positive_vectors) / len(positive_vectors)
    neg_score = sum(cosine_similarity(review_vec, v) for v in negative_vectors) / len(negative_vectors)

    if pos_score == 0 and neg_score == 0:
        return "UNSURE"   # the review shared no words with either group
    return "POSITIVE" if pos_score >= neg_score else "NEGATIVE"

print("✅ Meaning model ready.")

### Try the meaning model

Run this, then **change the sentence** and try your own. Run 10–15 sentences through it:
where does it work well, and what patterns tend to break it?

In [ ]:
review = "An exciting and enjoyable movie"   # 👉 CHANGE THIS

result = similarity_sentiment(review)
print(f'Review:  "{review}"')
print(f'Verdict: {result}')

## Step 3 — Head to head

Let's run **both models on the same reviews** and put their answers side by side.

Watch especially the **tricky** reviews at the bottom — the ones with *"not"*, sarcasm,
or words that aren't on our lists. Where do the two models disagree? Which one gets it right?

In [ ]:
test_reviews = [
    # Straightforward ones
    "A brilliant and wonderful film",
    "A boring and terrible waste of time",
    # Words NOT on the rule-based lists
    "An absolute masterpiece, utterly magnificent",
    "Tedious, clunky and instantly forgettable rubbish",
    # The tricky ones from your paper activity
    "This was not good at all",
    "Not bad actually, I really enjoyed it",
    "Oh great, another two hours I will never get back",
    "Far from boring, it was thrilling",
]

# Print a tidy comparison table.
print(f'{"REVIEW":<52} {"RULE-BASED":<12} {"MEANING":<10}')
print("-" * 76)
for review in test_reviews:
    r = rule_based_sentiment(review)
    s = similarity_sentiment(review)
    short = (review[:49] + "...") if len(review) > 52 else review
    print(f'{short:<52} {r:<12} {s:<10}')

### What to look for

Look down the two columns and find the rows where the models **disagree**.

- **"An absolute masterpiece..."** — the rule-based model says `UNSURE`, because none of those
  words are on its lists. Does the meaning model do any better?
- **"This was not good at all"** — both models see the word *"good"* and lean `POSITIVE`.
  Neither of them understands the word *"not"*! This is a big limitation of counting words.
- **"Oh great, another two hours I will never get back"** — pure sarcasm, with no negative
  words at all. This one is genuinely hard.

**Discussion:** Neither model truly *understands* language — they both work by matching words.
The meaning model is more flexible (it isn't limited to a fixed list), but it still can't handle
*"not"*, sarcasm, or word order. Real AI models try to fix this by learning from huge amounts of
text — which is what we'll look at in the next notebook.

## Step 4 — Your challenge

In the cell below, **write your own review** and see if you can:

1. Find a sentence where the two models **disagree**.
2. Find a sentence that **fools** both of them.
3. Try sarcasm, slang, or unusual words. What breaks them?

Change the sentence, run the cell, and keep experimenting.

In [ ]:
your_review = "Type your own film review here!"   # 👉 CHANGE THIS

print(f'Your review: "{your_review}"')
print(f'Rule-based says: {rule_based_sentiment(your_review)}')
print(f'Meaning says:    {similarity_sentiment(your_review)}')

### One more thing to try

The meaning model's answers depend entirely on the **example reviews** you give it in Step 2.
Go back and add a few of your own examples to `positive_examples` and `negative_examples`,
re-run that cell and the comparison — can you make the model noticeably better (or worse)?